In [ ]:
!pip uninstall shapely geopandas google-cloud-bigquery -y
!pip install shapely --no-deps
!pip install google-cloud-bigquery pandas pandas-gbq ollama

Found existing installation: shapely 2.1.2
Uninstalling shapely-2.1.2:
  Successfully uninstalled shapely-2.1.2
Found existing installation: google-cloud-bigquery 3.42.0
Uninstalling google-cloud-bigquery-3.42.0:
  Successfully uninstalled google-cloud-bigquery-3.42.0
  Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)
Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)
  Using cached google_cloud_bigquery-3.42.0-py3-none-any.whl.metadata (8.0 kB)
Using cached google_cloud_bigquery-3.42.0-py3-none-any.whl (263 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.42.0 requires geopandas>=0.12.2, which is not installed.


In [ ]:
import shapely
print(shapely.__version__)

2.1.2


In [ ]:
from google.colab import userdata
from google.cloud import bigquery
from pandas_gbq import to_gbq
import pandas as pd
from datetime import datetime
import ollama
import requests
import json

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [ ]:
from google.colab import userdata
from google.cloud import bigquery
from pandas_gbq import to_gbq
import pandas as pd
from datetime import datetime
import ollama

# Get the API key from Secrets
OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')

# Connect to BigQuery
client = bigquery.Client.from_service_account_json('/content/fintech-intelligence-platform-14a6170252ba.json')
print("✅ Successfully connected to BigQuery!")

# Query KPI
query = """
SELECT
    (SELECT COUNT(*) FROM `fintech-intelligence-platform.fip_dwh_gold.churn_predictions` WHERE risk_level = 'High') AS high_risk,
    (SELECT COUNT(*) FROM `fintech-intelligence-platform.fip_dwh_gold.churn_predictions` WHERE risk_level = 'Medium') AS medium_risk,
    (SELECT COUNT(*) FROM `fintech-intelligence-platform.fip_dwh_gold.churn_predictions` WHERE risk_level = 'Low') AS low_risk,
    (SELECT AVG(churn_probability) FROM `fintech-intelligence-platform.fip_dwh_gold.churn_predictions`) AS avg_churn_prob,
    (SELECT COUNT(*) FROM `fintech-intelligence-platform.fip_dwh_gold.dim_customers` WHERE customer_segment = 'Mass') AS mass_customers,
    (SELECT COUNT(*) FROM `fintech-intelligence-platform.fip_dwh_gold.dim_customers` WHERE churn_label = TRUE) AS total_churned
"""

df_kpi = client.query(query).to_dataframe()
print("📊 KPI Data:")
print(df_kpi)

✅ Successfully connected to BigQuery!
📊 KPI Data:
   high_risk  medium_risk  low_risk  avg_churn_prob  mass_customers  \
0       1493        18507     60000        0.179869           21436   

   total_churned  
0          14400  


In [ ]:
# Trích xuất giá trị
high_risk = df_kpi['high_risk'].iloc[0]
medium_risk = df_kpi['medium_risk'].iloc[0]
low_risk = df_kpi['low_risk'].iloc[0]
avg_prob = df_kpi['avg_churn_prob'].iloc[0]
mass_customers = df_kpi['mass_customers'].iloc[0]
total_churned = df_kpi['total_churned'].iloc[0]

# Tạo prompt
# Tạo prompt với format yêu cầu cụ thể
prompt = f"""
Bạn là một chuyên gia phân tích dữ liệu FinTech. Hãy viết một báo cáo ngắn (khoảng 100-150 từ) với 3 phần rõ ràng, sử dụng MARKDOWN.

Số liệu:
- Khách hàng High Risk: {high_risk} người
- Khách hàng Medium Risk: {medium_risk} người
- Khách hàng Low Risk: {low_risk} người
- Tỷ lệ churn trung bình: {avg_prob:.2%}
- Khách hàng phân khúc Mass: {mass_customers} người
- Tổng churn: {total_churned} người

Yêu cầu format (dùng markdown):
## 📊 Tình hình chung
(2-3 câu tóm tắt tình hình)

## ⚠️ Điểm đáng chú ý
(2-3 câu về điểm nổi bật)

## 💡 Khuyến nghị
(2-3 hành động cụ thể)
"""

# Gọi Ollama Cloud
client_ollama = ollama.Client(
    host="https://ollama.com",
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"}
)

response = client_ollama.chat(
    model="gpt-oss:120b-cloud",
    messages=[{"role": "user", "content": prompt}]
)

# Lấy nội dung AI
raw_insight = response["message"]["content"]
print("🤖 AI Generated Insight (raw):")
print(raw_insight)

🤖 AI Generated Insight (raw):
## 📊 Tình hình chung  
Trong tổng số 79 000 khách hàng, phân bố rủi ro là: 1 493 người High Risk, 18 507 người Medium Risk và 60 000 người Low Risk. Tỷ lệ churn trung bình đạt 17,99 % và đã tạo ra 14 400 khách hàng rời bỏ, chiếm khoảng 18,2 % tổng cơ sở.

## ⚠️ Điểm đáng chú ý  
- Đối tượng High Risk chiếm chỉ 1,9 % khách hàng nhưng đóng góp tỷ lệ churn cao hơn mức trung bình.  
- Phân khúc Mass (21 436 người) bao gồm phần lớn khách hàng Medium Risk, là nguồn tiềm năng cho các chiến dịch giữ chân.  

## 💡 Khuyến nghị  
1. Triển khai chương trình ưu đãi đặc biệt cho khách hàng High Risk (ví dụ: giảm phí, dịch vụ hỗ trợ cá nhân) để giảm churn.  
2. Áp dụng mô hình dự báo churn dựa trên hành vi giao dịch cho phân khúc Mass, nhằm xác định sớm các tín hiệu rời bỏ.  
3. Tăng cường chiến dịch giáo dục tài chính và khuyến mãi cho khách hàng Low Risk để duy trì mức churn ở mức tối thiểu.


In [ ]:
# Tách nội dung thành các phần (nếu AI trả về markdown)
def format_insight_text(text):
    """
    Định dạng lại insight để hiển thị đẹp trên dashboard.
    - Nếu có markdown heading (##) thì giữ nguyên
    - Nếu không, thêm line breaks
    """
    lines = text.strip().split('\n')
    formatted_lines = []

    for line in lines:
        line = line.strip()
        if line.startswith('##'):
            formatted_lines.append('')  # Thêm khoảng trống trước heading
            formatted_lines.append(line)
        elif line.startswith('-'):
            formatted_lines.append(f"  {line}")  # Thụt lề bullet points
        else:
            formatted_lines.append(line)

    return '\n'.join(formatted_lines)

formatted_insight = format_insight_text(raw_insight)

print("\n📝 Formatted Insight:")
print(formatted_insight)

# Tạo DataFrame
insight_df = pd.DataFrame([{
    'insight_date': datetime.now().strftime('%Y-%m-%d'),
    'insight_text': formatted_insight,
    'raw_text': raw_insight,  # Lưu cả bản gốc để tham khảo
    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}])

# Ghi vào BigQuery
to_gbq(
    insight_df,
    destination_table='fip_dwh_gold.daily_insights',
    project_id='fintech-intelligence-platform',
    if_exists='replace',
    credentials=client._credentials
)

print("✅ AI Insight saved to BigQuery!")


📝 Formatted Insight:

## 📊 Tình hình chung
Trong tổng số 79 000 khách hàng, phân bố rủi ro là: 1 493 người High Risk, 18 507 người Medium Risk và 60 000 người Low Risk. Tỷ lệ churn trung bình đạt 17,99 % và đã tạo ra 14 400 khách hàng rời bỏ, chiếm khoảng 18,2 % tổng cơ sở.


## ⚠️ Điểm đáng chú ý
  - Đối tượng High Risk chiếm chỉ 1,9 % khách hàng nhưng đóng góp tỷ lệ churn cao hơn mức trung bình.
  - Phân khúc Mass (21 436 người) bao gồm phần lớn khách hàng Medium Risk, là nguồn tiềm năng cho các chiến dịch giữ chân.


## 💡 Khuyến nghị
1. Triển khai chương trình ưu đãi đặc biệt cho khách hàng High Risk (ví dụ: giảm phí, dịch vụ hỗ trợ cá nhân) để giảm churn.
2. Áp dụng mô hình dự báo churn dựa trên hành vi giao dịch cho phân khúc Mass, nhằm xác định sớm các tín hiệu rời bỏ.
3. Tăng cường chiến dịch giáo dục tài chính và khuyến mãi cho khách hàng Low Risk để duy trì mức churn ở mức tối thiểu.


100%|██████████| 1/1 [00:00<00:00, 9868.95it/s]

✅ AI Insight saved to BigQuery!


In [ ]:
# Tạo DataFrame chứa insight
insight_df = pd.DataFrame([{
    'insight_date': datetime.now().strftime('%Y-%m-%d'),
    'insight_text': insight_text,
    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}])

# Ghi vào BigQuery
to_gbq(
    insight_df,
    destination_table='fip_dwh_gold.daily_insights',
    project_id='fintech-intelligence-platform',
    if_exists='replace',  # 'append' nếu muốn giữ lịch sử
    credentials=client._credentials
)

print("✅ AI Insight saved to BigQuery!")
print(f"📝 Insight date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"📄 First 100 characters: {insight_text[:100]}...")

100%|██████████| 1/1 [00:00<00:00, 7752.87it/s]

✅ AI Insight saved to BigQuery!
📝 Insight date: 2026-06-16
📄 First 100 characters: **1. Tình hình chung:** Tỷ lệ churn trung bình của toàn bộ cơ sở khách hàng đang ở mức 17.99%, tương...
